# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #1:

Claim: Content trending upward has a different structural profile than content trending downward.

My Methodology question: According to the research, growing pages are 37.6% longer with a higher quantity of words than declining pages. If someone were to edit a page in the past 30 days and it grew, would this change be the cause of growth or a result of growth?. Why does it matter?, according to the page's logic, "expand thin pages that already earn impressions:, in other words, more words = more growth. However, what if a page grows, and the editor is keen to add more changes and words in hopes of growing the page further. In this case, More Growth = More Words.

Finding #5:

Claim: High scroll + high engagement = +11.2 health points. Visibility consistency compounds the effect.

My Methodology question: Scroll makes up as one of the factors when calculating Health Score Formula. If scroll depth is summed up to make up the HSF, and finding #5 says : More Scroll = More visibility, and More Scroll = More Health, is that actually two independent things confirming each other? Or is one partly just measuring the other, by construction?. Why it matters? If scroll depth is already one of the four ingredients baked into Health Score, then "high scroll = high Health Score" isn't fully independent confirmation — it's partly the formula agreeing with itself. That matters because a reader could see "+11.2 health points" and think it reflects real-world impact (more engaged users, better rankings), when part of that number was mathematically guaranteed just by scroll depth being a direct input to the score.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
!pip install duckdb --quiet

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)
labeled['days_since_update'] = labeled['days_since_update'].fillna(9999)

features = ['march_impressions', 'march_clicks', 'march_avg_position', 'days_since_update']
print(f"Total pages: {len(labeled)}, base rate (declining): {labeled['declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(train_df, test_df, label=""):
    X_train, y_train = train_df[features], train_df['declining']
    X_test, y_test = test_df[features], test_df['declining']
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    base_rate = y_test.mean()
    print(f"--- {label} ---")
    print(f"Train pages: {len(train_df)}, Test pages: {len(test_df)}")
    print(f"Train base rate: {y_train.mean():.3f}, Test base rate: {base_rate:.3f}")
    for k in (20, 50):
        print(f"Precision@{k}: {precision_at_k(scores, y_test, k):.3f}  (base rate {base_rate:.3f})")
    print()
    return model, scores

In [ ]:
# BEFORE -- naive random split, ignoring that pages repeat within a client
rand_train, rand_test = train_test_split(labeled, test_size=0.3, random_state=42)
_ = fit_and_score(rand_train, rand_test, label="BEFORE: random row split")

# AFTER -- grouped by client_hash_id, same as w05
unique_clients = labeled['client_hash_id'].unique()
train_clients, test_clients = train_test_split(unique_clients, test_size=0.3, random_state=42)
grp_train = labeled[labeled['client_hash_id'].isin(train_clients)].copy()
grp_test = labeled[labeled['client_hash_id'].isin(test_clients)].copy()
_ = fit_and_score(grp_train, grp_test, label="AFTER: grouped-by-client split (w05 original)")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Train-without-suspect test on march_clicks (the top-magnitude coefficient)
# per the leakage taxonomy: a collapse from near-perfect to a real score is the confession;
# a small, proportionate drop is what we'd expect from removing one honest signal, not a leak.
reduced_features = ['march_impressions', 'march_avg_position', 'days_since_update']

def fit_and_score_features(train_df, test_df, feat_list, label=""):
    X_train, y_train = train_df[feat_list], train_df['declining']
    X_test, y_test = test_df[feat_list], test_df['declining']
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    print(f"--- {label} ---")
    for k in (20, 50):
        print(f"Precision@{k}: {precision_at_k(scores, y_test, k):.3f}")
    print()

fit_and_score_features(grp_train, grp_test, features, label="WITH march_clicks")
fit_and_score_features(grp_train, grp_test, reduced_features, label="WITHOUT march_clicks")

In [ ]:
# Verification per the skill: deliberately ADD a leaky feature and confirm the harness reacts.
# client_decline_rate is computed FROM the training label and joined back in by client --
# a textbook decision/label-derived leak if it were ever used for real.
client_rate = grp_train.groupby('client_hash_id')['declining'].mean().rename('client_decline_rate')

leaky_train = grp_train.merge(client_rate, on='client_hash_id', how='left')
leaky_test = grp_test.merge(client_rate, on='client_hash_id', how='left')
# test clients are unseen in train, so this mostly becomes NaN -> fill with train mean
leaky_train['client_decline_rate'] = leaky_train['client_decline_rate'].fillna(grp_train['declining'].mean())
leaky_test['client_decline_rate'] = leaky_test['client_decline_rate'].fillna(grp_train['declining'].mean())

leaky_features = features + ['client_decline_rate']
fit_and_score_features(leaky_train, leaky_test, leaky_features, label="WITH deliberate leak (client_decline_rate)")
fit_and_score_features(grp_train, grp_test, features, label="WITHOUT the leak (original 4 features)")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (w05, Section 3) — my boldest sentence:

"This is a real, robust finding, not an artifact of the leak — fixing the leak shifted both numbers but didn't change which method wins where."

Rewritten:

In this sample, the model measured higher precision@20 than the baseline (0.500 vs. 0.400), and the baseline measured higher precision@50 (0.480 vs. 0.380) — a pattern observed both before and after the days_since_update leak fix. This is directional evidence from one client-grouped 70/30 split on one month's data, not a validated guarantee that either method wins its respective k reliably on other periods or client mixes.

Also worth softening (w05, error analysis):

"...so the model may be partly learning 'successful pages regress' rather than a genuine decline signal specific to these pages."

Rewritten:

A plausible explanation, not a confirmed one: the three most-confident wrong predictions all combine strong March performance with unknown staleness, consistent with regression-to-the-mean. Testing this would need multi-month history per page, which the current March-only feature set doesn't have — an open question, not a settled cause.

New claim from this audit (fill in after Section 2 runs):

The random-split precision@k measured [X] vs. the grouped-split precision@k of [Y] on the same model and features. Read as an observed sensitivity check on split design for this dataset, not as proof of one specific leakage mechanism.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.